### Building A Chatbot

In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:


    Conversational RAG: Enable a chatbot experience over an external source of data
    Agents: Build a chatbot that can take actions


This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [136]:
import os 
from dotenv import load_dotenv
load_dotenv()

groq_api_key=os.getenv("GROQ_API_KEY")
groq_api_key

'YOUR_GROQ_API_KEY'

In [137]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.3-70b-versatile",groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x139754ef0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1392236b0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [138]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Aman and I am a Chief AI Engineer")])

AIMessage(content="Nice to meet you, Aman. It's great to connect with a Chief AI Engineer like yourself. What brings you here today? Are you working on any exciting AI projects or looking for assistance with a particular challenge? I'm here to help and learn from your expertise.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 49, 'total_tokens': 105, 'completion_time': 0.152861446, 'completion_tokens_details': None, 'prompt_time': 0.004486749, 'prompt_tokens_details': None, 'queue_time': 0.04783019, 'total_time': 0.157348195}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3851-4d69-7463-80b6-75f5daf7f730-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 56, 'total_tokens': 105})

In [139]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi, My name is Aman and I am a Chief AI Engineer"),
        AIMessage(content="Hello Aman, nice to meet you. It's great to connect with a Chief AI Engineer. That sounds like a fascinating and challenging role. What kind of projects are you currently working on, and what areas of AI are you most interested in?"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ] 
)

AIMessage(content="Your name is Aman, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 120, 'total_tokens': 135, 'completion_time': 0.032559353, 'completion_tokens_details': None, 'prompt_time': 0.049205948, 'prompt_tokens_details': None, 'queue_time': 0.236102016, 'total_time': 0.081765301}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3851-5053-7b83-9f76-1e243489c90c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 120, 'output_tokens': 15, 'total_tokens': 135})

### Message History

We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [140]:
from langchain_community.chat_message_histories import ChatMessageHistory 
from langchain_core.chat_history import BaseChatMessageHistory 
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [141]:
config={"configurable":{"session_id":"chat1"}}

In [142]:
with_message_history.invoke(
    [HumanMessage(content="Hi, my name is Aman and I am a Chief AI Engineer")],
    config=config
)

AIMessage(content="Nice to meet you, Aman. As a Chief AI Engineer, you must be working on some exciting and innovative projects. What type of AI applications are you currently focusing on, and what industries or domains are you applying them to? I'm here to chat and help with any questions or topics you'd like to discuss.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 49, 'total_tokens': 115, 'completion_time': 0.258602643, 'completion_tokens_details': None, 'prompt_time': 0.007929264, 'prompt_tokens_details': None, 'queue_time': 0.254755662, 'total_time': 0.266531907}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3851-52a2-7951-927f-6f826290a05d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 66, 'total_tokens': 115})

In [143]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi, my name is Aman and I am a Chief AI Engineer")],
    config=config
)

In [144]:
response.content

"Hello again, Aman. It's nice to see you again. As a Chief AI Engineer, you have a critical role in shaping the future of artificial intelligence and its applications. What are some of the most interesting or challenging projects you've worked on recently, and how do you see AI evolving in the next few years?"

In [145]:
with_message_history.invoke(
    [HumanMessage(content="What is my name ?")],
    config=config
)

AIMessage(content="Your name is Aman, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 218, 'total_tokens': 233, 'completion_time': 0.034422072, 'completion_tokens_details': None, 'prompt_time': 0.010457622, 'prompt_tokens_details': None, 'queue_time': 0.052773689, 'total_time': 0.044879694}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3851-58db-78b3-ade4-cfde2eb3bf4d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 218, 'output_tokens': 15, 'total_tokens': 233})

In [146]:
## change the config-- session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats is my name ?")],
    config=config1
)

response.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to know your personal details, including your name, unless you tell me. If you'd like to share your name, I'd be happy to chat with you and address you by your name."

In [147]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John ?")],
    config=config1
)

response.content

"Hello John. It's nice to meet you. I'll make sure to address you by your name in our conversation. How's your day going so far, John? Is there anything specific you'd like to talk about or ask me?"

In [148]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats is my name ?")],
    config=config1,
)

response.content

'I remember, your name is John. We established that earlier in our conversation.'

### Prompt templates

Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [149]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer the question to the best of your ability "),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [150]:
chain.invoke({"messages":[HumanMessage(content="Hi, My name is Aman Singh")]})

AIMessage(content="Hello Aman Singh, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 58, 'total_tokens': 86, 'completion_time': 0.052330516, 'completion_tokens_details': None, 'prompt_time': 0.006926354, 'prompt_tokens_details': None, 'queue_time': 1.648397518, 'total_time': 0.05925687}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3851-60c8-7483-b642-f6388cfb252a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 28, 'total_tokens': 86})

In [151]:
with_messsage_history=RunnableWithMessageHistory(chain,get_session_history)

In [152]:
config4={"configurable":{"session_id":"chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Aman Singh")],
    config=config4
)
response

AIMessage(content="Hello Aman Singh, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 43, 'total_tokens': 71, 'completion_time': 0.048013388, 'completion_tokens_details': None, 'prompt_time': 0.002234409, 'prompt_tokens_details': None, 'queue_time': 0.095881549, 'total_time': 0.050247797}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d3851-68d1-7e92-86a1-adf46a8dc2d3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 43, 'output_tokens': 28, 'total_tokens': 71})

In [153]:
response=with_message_history.invoke(
    [HumanMessage(content="What is my name?")],
    config=config4
)
response.content

'Your name is Aman Singh.'

In [154]:
## Add more complexity 
prompt=ChatPromptTemplate.from_messages(
    [ 
       (
           "system",
           "You are a helpful assistant.Answer the question to the best of your ability in {language}.",
       ),
       MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt|model

In [155]:
response=chain.invoke({"messages":[HumanMessage(content="Hi, My name is Aman Singh")],"language":"Hindi"})
response.content

'नमस्ते अमन सिंह, मैं आपकी मदद करने के लिए तैयार हूँ। मुझसे क्या पूछना चाहते हैं?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [156]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [157]:
config5={"configurable":{"session_id":"chat4"}}
response=with_message_history.invoke(
    {"messages":[HumanMessage(content="Hi, My name is Aman Singh")],"language":"Hindi"},
    config=config5,
)

response.content

'नमस्ते अमन सिंह जी, मैं आपकी कैसे मदद कर सकता हूँ?'

In [158]:
response=with_message_history.invoke(
    {"messages":[HumanMessage(content="what is my name?")],"language":"Hindi"},
    config=config5,
)

response.content

'आपका नाम अमन सिंह है।'

### Managing the Conversation History

One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

 'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [177]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
    
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [182]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model 

 )
response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="what ice cream do i like")],
    "language":"English"
    }
    
)
response.content

"I don't know, I'm a large language model, I don't have personal information about you, including your favorite ice cream flavor. But I can ask: What's your favorite ice cream flavor?"

In [183]:
response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="what math problem did i ask you")],
    "language":"English"
    }
    
)
response.content

'You asked me to solve the math problem: 2 + 2. The answer was 4.'

In [180]:
## Let wrap this in the message history 
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config={"configurable":{"session_id":"chat5"}}

In [181]:
response=with_message_history.invoke(
    {
    "messages":messages + [HumanMessage(content="what is my name?")],
    "language":"English"
    },
    config=config
    
)
response.content

"I don't know your name, you haven't told me yet. Would you like to share it with me?"